# HFP — Eğitilebilirlik Hipotezi Testi (B) — GPU

## Motivasyon
Streaming kararlılık testi (v2) beklenen "cubic daha kararlı" sonucunu vermedi —
ama **cubic aynı bütçede çok daha iyi öğrendi** (loss 1.098 vs 1.936). Önceki
ön-bulguyla tutarlı: ctx 512'de cubic 2/3 seed'de ln(vocab) platosunu kırdı,
exp sadece 1/3.

## Soru
Cubic, uzun bağlamda (ctx 256-512) exp'ten **daha kolay öğreniyor** mu?

## Deney Tasarımı
- Dense retention görevi (P=8, 8 sorgu/dizi)
- ctx ∈ {160, 256, 384, 512} — kademeli uzatma
- retention ∈ {exp, cubic_flux_chunked}, feature_map = dpfp
- 3 seed (0, 1, 2)
- 600 adım, lr = 1e-3, batch = 16
- **GPU: T4** (otomatik algılanır)
- Ölçüm: (1) son loss, (2) ln(vocab) platosunu kırdı mı (loss < 3.0)

## Önceden Yazılan Başarı Kriteri
> "cubic_flux_chunked, ctx ≥ 384'te 3 seed'in ≥2'sinde ln(vocab) platosunu kırar
> VE exp bunun 3 seed'in ≤1'inde başarırsa → cubic'in uzun-bağlam eğitilebilirlik
> avantajı DOĞRULANDI. Aksi halde hipotez reddedilir / sonuçsuz."

⚠️ **NaN riski:** Önceki ön-bulguda cubic bir kez NaN'ladı. NaN = o seed sayılmaz, not düşülür.

In [ ]:
# --- KURULUM ---
import subprocess, sys, os
REPO = "/kaggle/working/HFP"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/kayra-hn/HFP.git", REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.40"], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
os.environ["HFP_CKPT_DIR"] = "/kaggle/working/ck"
os.environ["PYTHONPATH"] = REPO
os.makedirs("/kaggle/working/ck", exist_ok=True)
import torch
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")
print("Kurulum tamam!")

In [ ]:
# --- EGITILEBIRLIK TESTI (GPU) ---
import torch, random, math, time, numpy as np
from hfp.models.configuration_hfp import HFPConfig
from hfp.models.modeling_hfp import HFPForCausalLM

DEV = "cuda" if torch.cuda.is_available() else "cpu"

# --- Sabitler ---
KLO, KHI, VLO, VHI, FHI = 100, 130, 130, 160, 100
WIN, P, ANS = 8, 8, 0
STEPS, LR, BS = 600, 1e-3, 16
CTX_LIST = [160, 256, 384, 512]
SEEDS = [0, 1, 2]
RET_MODES = ["exp", "cubic_flux_chunked"]
FMAP = "dpfp"
LN_VOCAB = math.log(VHI - VLO)  # ln(30) ≈ 3.40 — plato seviyesi
PLATO_THRESHOLD = 3.0           # bunun altına düşerse "plato kırıldı"

def make_seq(ctx):
    toks = [random.randint(1, FHI - 1) for _ in range(ctx)]
    slots = random.sample(range(ctx // 2), 2 * P)
    random.shuffle(slots)
    keys = random.sample(range(KLO, KHI), P)
    lab = [-100] * ctx
    for i in range(P):
        a, b = slots[2 * i], slots[2 * i + 1]
        if a > b: a, b = b, a
        wp, qp = 2 * a, 2 * b
        v = random.randint(VLO, VHI - 1)
        toks[wp] = keys[i]; toks[wp + 1] = v
        toks[qp] = keys[i]; toks[qp + 1] = ANS
        lab[qp + 1] = v
    return toks, lab

def build(ret, ctx):
    cfg = HFPConfig(
        vocab_size=VHI + 4, hidden_size=64, num_hidden_layers=2,
        num_attention_heads=2, intermediate_size=256, bulk_dim=32,
        short_len=8, max_position_embeddings=ctx + 8, local_window=WIN,
        decay_mode=ret, rec_block=32, write_rule="additive",
        key_feature_map=FMAP, ffn_type="standard"
    )
    return HFPForCausalLM(cfg).to(DEV)

def train_and_measure(ret, ctx, seed, budget=1500):
    """Eğit, son loss ve eğri döndür."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if DEV == "cuda": torch.cuda.manual_seed(seed)
    model = build(ret, ctx)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STEPS)
    model.train()
    t0 = time.time()
    loss_curve = []
    nan_flag = False
    for step in range(1, STEPS + 1):
        if time.time() - t0 > budget:
            print(f"    BUTCE ASILDI step {step}"); break
        xs, ys = [], []
        for _ in range(BS):
            t, l = make_seq(ctx); xs.append(t); ys.append(l)
        out = model(torch.tensor(xs, device=DEV), labels=torch.tensor(ys, device=DEV))
        if not torch.isfinite(out.loss):
            print(f"    NaN! step {step}")
            nan_flag = True; break
        opt.zero_grad(set_to_none=True)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sch.step()
        loss_val = out.loss.item()
        loss_curve.append(loss_val)
        if step % 200 == 0:
            print(f"    step {step}/{STEPS}  loss={loss_val:.3f}  ({time.time()-t0:.0f}s)", flush=True)
    final_loss = loss_curve[-1] if loss_curve else float('inf')
    broke_plateau = final_loss < PLATO_THRESHOLD and not nan_flag
    elapsed = time.time() - t0
    return {
        'final_loss': round(final_loss, 3),
        'broke_plateau': broke_plateau,
        'nan': nan_flag,
        'min_loss': round(min(loss_curve) if loss_curve else float('inf'), 3),
        'curve_200': [round(loss_curve[i-1], 3) for i in range(200, len(loss_curve)+1, 200)],
        'steps_done': len(loss_curve),
        'time_s': round(elapsed, 1)
    }

# ========== ANA DENEY ==========
all_results = {}
total_t0 = time.time()

for ctx in CTX_LIST:
    print(f"\n{'#'*60}")
    print(f"  CTX = {ctx}  (ln(vocab) = {LN_VOCAB:.2f}, plato esigi = {PLATO_THRESHOLD})")
    print(f"{'#'*60}")
    for ret in RET_MODES:
        for seed in SEEDS:
            tag = f"{ret}|ctx{ctx}|s{seed}"
            print(f"\n  --- {tag} ---")
            r = train_and_measure(ret, ctx, seed)
            all_results[tag] = r
            status = "NaN!" if r['nan'] else ("KIRDI" if r['broke_plateau'] else "PLATO")
            print(f"  => loss={r['final_loss']}  min={r['min_loss']}  [{status}]  ({r['time_s']}s)")
            print(f"     egri(200): {r['curve_200']}")

total_elapsed = time.time() - total_t0
print(f"\nToplam sure: {total_elapsed/60:.1f} dakika")

# ========== SONUC TABLOSU ==========
print(f"\n\n{'='*75}")
print("EGITILEBIRLIK SONUCLARI")
print(f"{'='*75}")
print(f"{'Mod':25s} {'ctx':>5s} {'seed':>5s} {'son loss':>10s} {'min loss':>10s} {'sure':>8s} {'durum':>8s}")
print("-"*75)
for tag, r in sorted(all_results.items()):
    parts = tag.split('|')
    ret, ctx_s, seed_s = parts[0], parts[1], parts[2]
    status = "NaN!" if r['nan'] else ("KIRDI" if r['broke_plateau'] else "PLATO")
    print(f"{ret:25s} {ctx_s:>5s} {seed_s:>5s} {r['final_loss']:10.3f} {r['min_loss']:10.3f} {r['time_s']:7.0f}s {status:>8s}")

# ========== KARAR KRITERI ==========
print(f"\n\n{'='*75}")
print("KARAR KRITERI DEGERLENDIRMESI")
print(f"{'='*75}")
for ctx in CTX_LIST:
    print(f"\nctx = {ctx}:")
    for ret in RET_MODES:
        broke = sum(1 for s in SEEDS
                    if all_results.get(f"{ret}|ctx{ctx}|s{s}", {}).get('broke_plateau', False))
        nans = sum(1 for s in SEEDS
                   if all_results.get(f"{ret}|ctx{ctx}|s{s}", {}).get('nan', False))
        valid = len(SEEDS) - nans
        losses = [all_results[f"{ret}|ctx{ctx}|s{s}"]['final_loss']
                  for s in SEEDS if not all_results[f"{ret}|ctx{ctx}|s{s}"]['nan']]
        mean_l = round(np.mean(losses), 3) if losses else float('inf')
        print(f"  {ret:25s}: plato kiran = {broke}/{valid} (NaN: {nans})  ort loss = {mean_l}")

print("\n" + "="*75)
print("KRITER: ctx >= 384'te cubic >= 2/3 kiriyorsa VE exp <= 1/3")
print("kiriyorsa -> egitilebirlik avantaji DOGRULANDI.")
print("="*75)

In [ ]:
# --- OZET GRAFIK ---
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, len(CTX_LIST), figsize=(5*len(CTX_LIST), 5), sharey=True)
colors = {'exp': '#e74c3c', 'cubic_flux_chunked': '#2980b9'}

for ci, ctx in enumerate(CTX_LIST):
    ax = axes[ci]
    for ret in RET_MODES:
        losses = []
        for seed in SEEDS:
            tag = f"{ret}|ctx{ctx}|s{seed}"
            r = all_results.get(tag, {})
            if not r.get('nan', True):
                losses.append(r['final_loss'])
        if losses:
            mean_loss = np.mean(losses)
            ax.bar(ret.replace('cubic_flux_chunked', 'cubic'), mean_loss,
                   color=colors[ret], alpha=0.7, edgecolor='black')
            for j, l in enumerate(losses):
                ax.scatter(ret.replace('cubic_flux_chunked', 'cubic'), l,
                          color='black', s=30, zorder=5)
    ax.axhline(y=PLATO_THRESHOLD, color='gray', linestyle='--', alpha=0.5, label=f'plato esigi ({PLATO_THRESHOLD})')
    ax.axhline(y=LN_VOCAB, color='orange', linestyle=':', alpha=0.5, label=f'ln(vocab) ({LN_VOCAB:.1f})')
    ax.set_title(f'ctx = {ctx}', fontsize=13)
    ax.set_ylabel('Final Loss' if ci == 0 else '', fontsize=12)
    ax.set_ylim(0, LN_VOCAB + 0.5)
    if ci == 0:
        ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2, axis='y')

fig.suptitle('Egitilebirlik: exp vs cubic (final loss by context length)', fontsize=14, y=1.02)
fig.tight_layout()
plt.savefig('/kaggle/working/trainability_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figur kaydedildi!")

## Dürüstlük Notları

1. **NaN durumunda:** O seed sayılmaz, "NaN" olarak raporlanır, gizlenmez.
2. **Eğer iki mod da öğrenemezse:** "Bu bütçede ctx ≥ X'te iki mod da plato kıramadı" → bulgu: eğitilebilirlik darboğazı ctx'e bağlı, moda değil.
3. **Eğer exp de iyi öğrenirse:** "İki mod arasında eğitilebilirlik farkı yok" → hipotez reddedilir.
4. Bu test **tek LR** (1e-3) kullanıyor. Önceki deneyler bunu her iki mod için çalışan LR olarak gösterdi, ama LR duyarlılığı bir sonraki adımda kontrol edilmeli.
5. **Streaming v2'deki loss farkı (1.098 vs 1.936) bu testin motivasyonu** — ama o test ctx=160'taydı ve tek seed'di. Bu test çoklu ctx + 3 seed ile genelleyecek.